# TF-IDF Student Manual - Chapter 8 Practice Exercises

This notebook completes the Chapter 8 exercises from `bts/docs/TF-IDF_Student_Manual.pdf` using the local RACE project data. The manual refers to `val.csv`; in this project the validation split is named `dev.csv`, so `dev.csv` is used wherever the exercise says validation/val.

In [1]:
from pathlib import Path
import itertools
import math
import re
import textwrap
import warnings

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.metrics.pairwise import cosine_similarity, paired_cosine_distances
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"
OPTION_COLUMNS = ["A", "B", "C", "D"]
TOKEN_RE = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")
SENTENCE_RE = re.compile(r"(?<=[.!?])\s+")


def tokenize(text):
    return TOKEN_RE.findall(str(text).lower())


def remove_stopwords(text, keep_question_cues=False):
    stop_words = set(ENGLISH_STOP_WORDS)
    if keep_question_cues:
        stop_words -= {"who", "what", "where", "when", "why", "how", "which", "many", "much"}
    return [token for token in tokenize(text) if token not in stop_words]


def normalize(text):
    return " ".join(tokenize(text))


def split_sentences(text):
    parts = SENTENCE_RE.split(str(text).replace("\n", " "))
    return [part.strip() for part in parts if part.strip()]


def correct_answer_text(row):
    if "correct_answer_text" in row and pd.notna(row["correct_answer_text"]):
        return str(row["correct_answer_text"])
    answer_letter = str(row.get("answer", "")).strip()
    if answer_letter in OPTION_COLUMNS:
        return str(row.get(answer_letter, ""))
    return ""


def jaccard(left, right):
    left_tokens = set(tokenize(left))
    right_tokens = set(tokenize(right))
    if not left_tokens or not right_tokens:
        return 0.0
    return len(left_tokens & right_tokens) / len(left_tokens | right_tokens)


def row_cosine(left_matrix, right_matrix):
    return 1.0 - paired_cosine_distances(left_matrix, right_matrix)

print("Project root:", PROJECT_ROOT)
print("Data files:", sorted(path.name for path in DATA_DIR.glob("*.csv")))

Project root: /home/shaffan/Desktop/Uni/AI/Project
Data files: ['dev.csv', 'test.csv', 'train.csv']


## Exercise 1 - Manual TF-IDF Calculation

We use the manual's normalized TF formula and the unsmoothed IDF formula `ln(N / df)` from Chapter 2. A smoothed scikit-learn-style IDF is also shown for reference.

In [2]:
sentences = {
    "S1": "The ancient Silk Road connected China to Europe.",
    "S2": "Merchants traded silk, spices, and glass along the route.",
    "S3": "The route passed through deserts and mountains.",
}

rows = []
for label, sentence in sentences.items():
    tokens = remove_stopwords(sentence)
    route_count = tokens.count("route")
    tf_route = route_count / len(tokens) if tokens else 0.0
    rows.append({
        "sentence": label,
        "tokens_after_stopword_removal": tokens,
        "token_count": len(tokens),
        "route_count": route_count,
        "TF(route)": tf_route,
    })

manual_df = pd.DataFrame(rows)
N = len(sentences)
df_route = sum("route" in set(remove_stopwords(sentence)) for sentence in sentences.values())
idf_route = math.log(N / df_route)
idf_route_smoothed = math.log((N + 1) / (df_route + 1)) + 1
manual_df["IDF(route), unsmoothed"] = idf_route
manual_df["TF-IDF(route), unsmoothed"] = manual_df["TF(route)"] * idf_route
manual_df["IDF(route), sklearn-smoothed"] = idf_route_smoothed
manual_df["TF-IDF(route), sklearn-smoothed"] = manual_df["TF(route)"] * idf_route_smoothed
manual_df

,sentence,tokens_after_stopword_removal,token_count,route_count,TF(route),"IDF(route), unsmoothed","TF-IDF(route), unsmoothed","IDF(route), sklearn-smoothed","TF-IDF(route), sklearn-smoothed"
0,S1,"[ancient, silk, road, connected, china, europe]",6,0,0.000000,0.405465,0.000000,1.287682,0.000000
1,S2,"[merchants, traded, silk, spices, glass, route]",6,1,0.166667,0.405465,0.067578,1.287682,0.214614
2,S3,"[route, passed, deserts, mountains]",4,1,0.250000,0.405465,0.101366,1.287682,0.321921


**Answer:** `route` appears once in both S2 and S3, and it appears in 2 of the 3 sentences. With `ln(N / df)`, `IDF(route) = ln(3/2) = 0.4055`. S3 has the higher TF-IDF score because, after stopword removal, S3 is shorter: `route` is 1 out of 4 tokens in S3 but only 1 out of 6 tokens in S2.

## Exercise 2 - scikit-learn Vectorization on RACE

We fit `TfidfVectorizer` on the training articles only, then transform the validation/dev articles using the same fitted vectorizer.

In [3]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
dev_df = pd.read_csv(DATA_DIR / "dev.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

article_vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words="english",
    sublinear_tf=True,
)
X_train_articles = article_vectorizer.fit_transform(train_df["article"].fillna(""))
X_dev_articles = article_vectorizer.transform(dev_df["article"].fillna(""))

print("Train rows:", len(train_df))
print("Dev rows:", len(dev_df))
print("Training TF-IDF matrix shape:", X_train_articles.shape)
print("Dev TF-IDF matrix shape after transform, no refit:", X_dev_articles.shape)

Train rows: 87866
Dev rows: 4887
Training TF-IDF matrix shape: (87866, 10000)
Dev TF-IDF matrix shape after transform, no refit: (4887, 10000)


In [4]:
feature_names = article_vectorizer.get_feature_names_out()
avg_scores = np.asarray(X_train_articles.mean(axis=0)).ravel()
top_average_idx = np.argsort(avg_scores)[::-1][:10]

top_average_terms = pd.DataFrame({
    "term": feature_names[top_average_idx],
    "average_tfidf": avg_scores[top_average_idx],
})
top_average_terms

,term,average_tfidf
0,people,0.025516
1,said,0.021242
2,time,0.019953
3,like,0.018617
4,day,0.017685
5,school,0.016754
6,good,0.016073
7,new,0.015689
8,make,0.015431
9,years,0.014880


In [5]:
article_index = 100
doc_vector = X_train_articles[article_index]
nonzero = doc_vector.tocoo()
order = np.argsort(nonzero.data)[::-1][:5]

top_doc_terms = pd.DataFrame({
    "term": feature_names[nonzero.col[order]],
    "tfidf": nonzero.data[order],
})
print("Article index:", article_index)
print(textwrap.shorten(train_df.loc[article_index, "article"], width=500, placeholder=" ..."))
top_doc_terms

Article index: 100
Tom was nine years old and he went to a school near his house. He went there on foot and came back home usually at 4 o'clock. But last Monday he was very late. His mother asked, "Why are you so late today, Tom?" "Because my teacher asked me to stay behind " Tom answered. "Why did the teacher make you stay behind?" the mother asked again. "Because no one could answer the teacher's question except me," Tom said. "What was the question?" his mother asked. "The question was 'Who broke the window ...


,term,tfidf
0,tom,0.476612
1,question,0.312000
2,teacher,0.273503
3,asked,0.249713
4,mother,0.245700


## Exercise 3 - Cosine Similarity

We reuse the vectorizer from Exercise 2. Cosine similarity compares the direction of TF-IDF vectors, so it can identify articles or sentences that use similar important words.

In [6]:
unique_test_articles = test_df.drop_duplicates("article").reset_index(drop=True)
article_sample = unique_test_articles.sample(n=3, random_state=7).reset_index(drop=True)
sample_vectors = article_vectorizer.transform(article_sample["article"].fillna(""))
similarity_matrix = cosine_similarity(sample_vectors)

pd.DataFrame(
    similarity_matrix,
    index=[f"article_{i}" for i in range(3)],
    columns=[f"article_{i}" for i in range(3)],
)

,article_0,article_1,article_2
article_0,1.000000,0.001996,0.010467
article_1,0.001996,1.000000,0.100193
article_2,0.010467,0.100193,1.000000


In [7]:
mask = np.triu(np.ones_like(similarity_matrix, dtype=bool), k=1)
pair_positions = np.argwhere(mask)
best_pair = pair_positions[np.argmax(similarity_matrix[mask])]
best_i, best_j = map(int, best_pair)

print(f"Most similar pair: article_{best_i} and article_{best_j}")
print(f"Cosine similarity: {similarity_matrix[best_i, best_j]:.4f}")
print("\nArticle", best_i, "snippet:")
print(textwrap.fill(textwrap.shorten(article_sample.loc[best_i, "article"], width=700, placeholder=" ..."), width=100))
print("\nArticle", best_j, "snippet:")
print(textwrap.fill(textwrap.shorten(article_sample.loc[best_j, "article"], width=700, placeholder=" ..."), width=100))

shared_terms = set(remove_stopwords(article_sample.loc[best_i, "article"])) & set(remove_stopwords(article_sample.loc[best_j, "article"]))
print("\nShared content terms, sample:", sorted(list(shared_terms))[:30])

Most similar pair: article_1 and article_2
Cosine similarity: 0.1002

Article 1 snippet:
My mother used to say things just to make me mad. Like most teenagers, I thought I knew everything.
And what I didn't know, I didn't want to be told. For example, if I said I was going to a movie, my
mother would roll her eyes. On my way out, she'd shout, " _ You don't want to learn that the hard
way!" I didn't know what that meant. I thought she just wanted me to stay at home. Many years later
I had three teenagers of my own. They thought they knew everything. I would say the same things to
them like my mom. That's when I first saw it. My mother wasn't trying to help me. Why do we always
have to learn things "the hard way"? Why can't we just accept our elders' wisdom? It's a good lesson
...

Article 2 snippet:
The word, "photography", was first used in 1839. It comes from the Greek words that mean "to write
with light". But photography could only give people _ pictures. So scientists were trying h

**Explanation:** the highest-scoring pair is the pair with the largest cosine value in the 3 by 3 matrix. The printed snippets and shared content terms show the vocabulary overlap that made those two articles closer in TF-IDF space.

In [8]:
row0 = test_df.iloc[0]
sentences0 = split_sentences(row0["article"])
question0 = row0["question"]
correct0 = correct_answer_text(row0)

sentence_vectors = article_vectorizer.transform(sentences0)
question_vector = article_vectorizer.transform([question0])
sentence_scores = cosine_similarity(sentence_vectors, question_vector).ravel()
top_sentence_order = np.argsort(sentence_scores)[::-1][:3]

print("Question:", question0)
print("Correct answer:", correct0)
print("\nTop 3 sentences by cosine similarity to the question:")
for rank, sentence_idx in enumerate(top_sentence_order, start=1):
    sentence = sentences0[sentence_idx]
    contains_answer = normalize(correct0) in normalize(sentence)
    print(f"{rank}. score={sentence_scores[sentence_idx]:.4f} | contains answer={contains_answer}")
    print(textwrap.fill(sentence, width=110))

Question: A discipline leader is supposed to _ .
Correct answer: make sure that nobody chats in class

Top 3 sentences by cosine similarity to the question:
1. score=0.6895 | contains answer=False
And there is a discipline leader who makes sure that nobody chats in class.
2. score=0.4321 | contains answer=False
But now I can ask the team leader or study leader.
3. score=0.3164 | contains answer=False
But being a team leader means you have to talk a lot.


In [9]:
rng = np.random.default_rng(42)
sample_indices = rng.choice(len(test_df), size=50, replace=False)

hit_rows = []
for idx in sample_indices:
    row = test_df.iloc[int(idx)]
    answer = correct_answer_text(row)
    sentences = split_sentences(row["article"])
    if not sentences or not normalize(answer):
        continue
    sentence_vectors = article_vectorizer.transform(sentences)
    question_vector = article_vectorizer.transform([row["question"]])
    scores = cosine_similarity(sentence_vectors, question_vector).ravel()
    order = np.argsort(scores)[::-1]
    top1_sentence = sentences[order[0]]
    top3_sentences = [sentences[i] for i in order[:3]]
    answer_norm = normalize(answer)
    hit_rows.append({
        "question_id": row["question_id"],
        "top1_hit": answer_norm in normalize(top1_sentence),
        "top3_hit": any(answer_norm in normalize(sentence) for sentence in top3_sentences),
    })

hit_df = pd.DataFrame(hit_rows)
print("Evaluated samples:", len(hit_df))
print("Top-1 sentence contains correct answer hit rate:", round(hit_df["top1_hit"].mean(), 4))
print("Top-3 sentence contains correct answer hit rate:", round(hit_df["top3_hit"].mean(), 4))
hit_df.head(10)

Evaluated samples: 50
Top-1 sentence contains correct answer hit rate: 0.1
Top-3 sentence contains correct answer hit rate: 0.12


,question_id,top1_hit,top3_hit
0,high13637.txt__q2,False,False
1,high17189.txt__q2,False,False
2,high15236.txt__q0,False,False
3,middle3233.txt__q2,False,False
4,high2934.txt__q2,False,False
5,high9122.txt__q3,False,False
6,high21028.txt__q2,False,False
7,high22824.txt__q4,False,False
8,middle8066.txt__q1,False,False
9,high15154.txt__q4,False,False


## Exercise 4 - Build the Full Feature Pipeline

Each RACE question becomes four option rows. The label is 1 for the correct option and 0 for the other three options, so `y_train.mean()` should be close to 0.25.

The sparse TF-IDF block uses `article + article + question + option`, following the manual's recommendation to give the article extra weight. Six dense cosine features are appended:

1. question-option cosine
2. article-option cosine
3. article-question cosine
4. article+question vs option cosine
5. article vs question+option cosine
6. question vs article+option cosine

In [10]:
def to_option_rows(frame):
    records = []
    for row_index, row in frame.reset_index(drop=True).iterrows():
        answer_letter = str(row.get("answer", "")).strip()
        for option_letter in OPTION_COLUMNS:
            option_text = str(row.get(option_letter, ""))
            records.append({
                "row_index": row_index,
                "question_id": row.get("question_id", row_index),
                "article": str(row.get("article", "")),
                "question": str(row.get("question", "")),
                "option_letter": option_letter,
                "option_text": option_text,
                "is_correct": int(option_letter == answer_letter),
                "combined_text": f"{row.get('article', '')} {row.get('article', '')} {row.get('question', '')} {option_text}",
            })
    return pd.DataFrame(records)


def build_verification_features(frame, vectorizer=None, fit=False):
    option_frame = to_option_rows(frame)
    if vectorizer is None:
        vectorizer = TfidfVectorizer(
            max_features=10000,
            stop_words="english",
            sublinear_tf=True,
            ngram_range=(1, 2),
            min_df=2,
        )
    if fit:
        tfidf_block = vectorizer.fit_transform(option_frame["combined_text"])
    else:
        tfidf_block = vectorizer.transform(option_frame["combined_text"])

    article_matrix = vectorizer.transform(option_frame["article"])
    question_matrix = vectorizer.transform(option_frame["question"])
    option_matrix = vectorizer.transform(option_frame["option_text"])
    article_question_matrix = vectorizer.transform(option_frame["article"] + " " + option_frame["question"])
    question_option_matrix = vectorizer.transform(option_frame["question"] + " " + option_frame["option_text"])
    article_option_matrix = vectorizer.transform(option_frame["article"] + " " + option_frame["option_text"])

    cosine_block = np.column_stack([
        row_cosine(question_matrix, option_matrix),
        row_cosine(article_matrix, option_matrix),
        row_cosine(article_matrix, question_matrix),
        row_cosine(article_question_matrix, option_matrix),
        row_cosine(article_matrix, question_option_matrix),
        row_cosine(question_matrix, article_option_matrix),
    ])
    cosine_names = [
        "question_option_cosine",
        "article_option_cosine",
        "article_question_cosine",
        "article_question_to_option_cosine",
        "article_to_question_option_cosine",
        "question_to_article_option_cosine",
    ]
    X = hstack([tfidf_block, csr_matrix(cosine_block)]).tocsr()
    y = option_frame["is_correct"].to_numpy()
    return X, y, option_frame, vectorizer, cosine_block, cosine_names

train_10k = train_df.head(10000).copy()
X_train_verify, y_train_verify, train_option_rows, verify_vectorizer, train_cosines, cosine_names = build_verification_features(
    train_10k,
    fit=True,
)
X_dev_verify, y_dev_verify, dev_option_rows, _, dev_cosines, _ = build_verification_features(
    dev_df,
    vectorizer=verify_vectorizer,
    fit=False,
)

print("X_train shape:", X_train_verify.shape)
print("X_dev shape:", X_dev_verify.shape)
print("y_train mean/class balance:", round(float(y_train_verify.mean()), 4))
print("Option rows per train question:", len(train_option_rows) / len(train_10k))

X_train shape: (40000, 10006)
X_dev shape: (19548, 10006)
y_train mean/class balance: 0.25
Option rows per train question: 4.0


In [11]:
clf = LogisticRegression(max_iter=1000, solver="liblinear", class_weight="balanced", random_state=42)
clf.fit(X_train_verify, y_train_verify)

dev_pred = clf.predict(X_dev_verify)
metrics = {
    "accuracy": accuracy_score(y_dev_verify, dev_pred),
    "macro_f1": f1_score(y_dev_verify, dev_pred, average="macro"),
}
print("Logistic Regression on TF-IDF + cosine features")
print({key: round(value, 4) for key, value in metrics.items()})
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_dev_verify, dev_pred))

Logistic Regression on TF-IDF + cosine features
{'accuracy': 0.5698, 'macro_f1': 0.5153}
Confusion matrix [[TN, FP], [FN, TP]]:
[[8846 5815]
 [2595 2292]]


In [12]:
# Compare TF-IDF-only and cosine-only ablations to investigate feature importance.
tfidf_feature_count = X_train_verify.shape[1] - len(cosine_names)

clf_tfidf_only = LogisticRegression(max_iter=1000, solver="liblinear", class_weight="balanced", random_state=42)
clf_tfidf_only.fit(X_train_verify[:, :tfidf_feature_count], y_train_verify)
tfidf_pred = clf_tfidf_only.predict(X_dev_verify[:, :tfidf_feature_count])

clf_cosine_only = LogisticRegression(max_iter=1000, solver="liblinear", class_weight="balanced", random_state=42)
clf_cosine_only.fit(train_cosines, y_train_verify)
cosine_pred = clf_cosine_only.predict(dev_cosines)

ablation = pd.DataFrame([
    {"feature_set": "TF-IDF only", "accuracy": accuracy_score(y_dev_verify, tfidf_pred), "macro_f1": f1_score(y_dev_verify, tfidf_pred, average="macro")},
    {"feature_set": "6 cosine features only", "accuracy": accuracy_score(y_dev_verify, cosine_pred), "macro_f1": f1_score(y_dev_verify, cosine_pred, average="macro")},
    {"feature_set": "TF-IDF + 6 cosine features", "accuracy": metrics["accuracy"], "macro_f1": metrics["macro_f1"]},
])

cosine_coef = pd.DataFrame({
    "feature": cosine_names,
    "coefficient": clf.coef_[0][-len(cosine_names):],
    "abs_coefficient": np.abs(clf.coef_[0][-len(cosine_names):]),
}).sort_values("abs_coefficient", ascending=False)

print("Ablation comparison:")
display(ablation)
print("Cosine feature coefficients in the combined model:")
display(cosine_coef)
print("Mean |coefficient| for TF-IDF block:", round(float(np.abs(clf.coef_[0][:tfidf_feature_count]).mean()), 6))
print("Mean |coefficient| for cosine block:", round(float(np.abs(clf.coef_[0][-len(cosine_names):]).mean()), 6))

Ablation comparison:


,feature_set,accuracy,macro_f1
0,TF-IDF only,0.524657,0.479462
1,6 cosine features only,0.580059,0.518995
2,TF-IDF + 6 cosine features,0.569777,0.515290


Cosine feature coefficients in the combined model:


,feature,coefficient,abs_coefficient
3,article_question_to_option_cosine,2.091446,2.091446
0,question_option_cosine,-1.932253,1.932253
5,question_to_article_option_cosine,1.369115,1.369115
1,article_option_cosine,0.986348,0.986348
4,article_to_question_option_cosine,-0.614705,0.614705
2,article_question_cosine,-0.251079,0.251079


Mean |coefficient| for TF-IDF block: 0.068474
Mean |coefficient| for cosine block: 1.207491


**Feature importance conclusion:** the ablation table shows how much performance comes from the sparse TF-IDF block versus the six cosine features alone. The coefficient table shows which cosine similarities the combined Logistic Regression relies on most strongly.

## Exercise 5 - Distractor Generation

This simple TF-IDF distractor generator retrieves question-relevant article sentences, extracts content-word phrases, filters phrases that are too similar to the correct answer, and keeps diverse candidates.

In [13]:
def candidate_phrases(sentence, max_ngram=3):
    tokens = [token for token in tokenize(sentence) if token not in ENGLISH_STOP_WORDS and len(token) > 2]
    phrases = []
    for n in range(1, max_ngram + 1):
        for start in range(0, max(len(tokens) - n + 1, 0)):
            phrase = " ".join(tokens[start:start + n])
            if len(phrase) >= 4:
                phrases.append(phrase)
    return phrases


def get_distractor_candidates(article, question, correct_answer, vectorizer, k=3):
    sentences = split_sentences(article)
    if not sentences:
        return []
    question_vector = vectorizer.transform([question])
    sentence_vectors = vectorizer.transform(sentences)
    sentence_scores = cosine_similarity(sentence_vectors, question_vector).ravel()
    correct_norm = normalize(correct_answer)
    correct_vector = vectorizer.transform([correct_answer])

    candidates = []
    for sentence_index in np.argsort(sentence_scores)[::-1]:
        sentence = sentences[int(sentence_index)]
        sentence_score = sentence_scores[int(sentence_index)]
        for phrase in candidate_phrases(sentence):
            phrase_norm = normalize(phrase)
            if not phrase_norm or phrase_norm == correct_norm:
                continue
            if correct_norm and (phrase_norm in correct_norm or correct_norm in phrase_norm):
                continue
            if jaccard(phrase, correct_answer) > 0.55:
                continue
            phrase_vector = vectorizer.transform([phrase])
            question_sim = cosine_similarity(phrase_vector, question_vector)[0, 0]
            correct_sim = cosine_similarity(phrase_vector, correct_vector)[0, 0]
            score = 0.45 * sentence_score + 0.45 * question_sim - 0.35 * correct_sim + 0.02 * min(len(phrase.split()), 3)
            candidates.append({
                "text": phrase,
                "score": float(score),
                "source_sentence": sentence,
            })

    # Keep the best scoring version of each normalized phrase.
    best_by_phrase = {}
    for candidate in candidates:
        key = normalize(candidate["text"])
        if key not in best_by_phrase or candidate["score"] > best_by_phrase[key]["score"]:
            best_by_phrase[key] = candidate

    selected = []
    for candidate in sorted(best_by_phrase.values(), key=lambda item: item["score"], reverse=True):
        if any(jaccard(candidate["text"], item["text"]) > 0.50 for item in selected):
            continue
        selected.append(candidate)
        if len(selected) == k:
            break
    return selected


def gold_distractors(row):
    answer_letter = str(row.get("answer", "")).strip()
    return [str(row[col]) for col in OPTION_COLUMNS if col != answer_letter and pd.notna(row[col])]


def bleu1_against_gold(generated, gold):
    smoothie = SmoothingFunction().method1
    gold_tokens = [tokenize(option) for option in gold if tokenize(option)]
    if not gold_tokens or not tokenize(generated):
        return 0.0
    return max(sentence_bleu([reference], tokenize(generated), weights=(1, 0, 0, 0), smoothing_function=smoothie) for reference in gold_tokens)


def mean_pairwise_cosine_distance(texts, vectorizer):
    if len(texts) < 2:
        return np.nan
    matrix = vectorizer.transform(texts)
    sims = cosine_similarity(matrix)
    distances = []
    for i, j in itertools.combinations(range(len(texts)), 2):
        distances.append(1.0 - sims[i, j])
    return float(np.mean(distances)) if distances else np.nan

sample_20 = test_df.sample(n=20, random_state=21).reset_index(drop=True)
results = []
for sample_number, row in sample_20.iterrows():
    correct = correct_answer_text(row)
    generated = get_distractor_candidates(row["article"], row["question"], correct, article_vectorizer, k=3)
    generated_texts = [item["text"] for item in generated]
    gold = gold_distractors(row)
    results.append({
        "sample": sample_number + 1,
        "question": row["question"],
        "correct_answer": correct,
        "generated_distractors": generated_texts,
        "gold_wrong_options": gold,
        "mean_bleu1_to_gold": float(np.mean([bleu1_against_gold(text, gold) for text in generated_texts])) if generated_texts else 0.0,
        "pairwise_cosine_distance": mean_pairwise_cosine_distance(generated_texts, article_vectorizer),
    })

distractor_df = pd.DataFrame(results)
for _, row in distractor_df.iterrows():
    print(f"Sample {row['sample']}: {row['question']}")
    print("  Correct:", row["correct_answer"])
    print("  Generated distractors:", row["generated_distractors"])
    print("  Original wrong options:", row["gold_wrong_options"])
    print()

print("Average BLEU-1 vs original wrong options:", round(distractor_df["mean_bleu1_to_gold"].mean(), 4))
print("Average pairwise cosine distance among generated distractors:", round(distractor_df["pairwise_cosine_distance"].mean(), 4))

Sample 1: Why does the author mention Kodak's invention of the first digital camera?
  Correct: To show its early attempt to reinvent itself.
  Generated distractors: ['kodak invented digital', "kodak's downfall", 'ago kodak']
  Original wrong options: ['To show its effort to overcome complacency.', 'To show its quick adaptation to the digital revolution.', "To show its will to compete with Japan's Fuji photo."]

Sample 2: The main idea of this passage is about _ .
  Correct: the development of kitesurfing
  Generated distractors: ['china xiamen place', 'xiamen place kitesurfing', 'place kitesurfing club']
  Original wrong options: ['the way of operating kitesurfing', 'the progress of kitesurfing equipment', 'the history of kitesurfing in China']

Sample 3: Among all the characters mentioned in the passage, who directed films in Hollywood?
  Correct: Burt and Verona.
  Generated distractors: ['films', 'films kill time', 'wonderful films']
  Original wrong options: ['Roland Emmerich.', 

In [14]:
manual_ratings = pd.DataFrame([
    {"sample": 1, "plausibility_1_to_5": 4, "note": "Kodak-related candidates are on-topic and believable, though phrase-like."},
    {"sample": 2, "plausibility_1_to_5": 3, "note": "Kitesurfing/location phrases are relevant but not polished answer choices."},
    {"sample": 3, "plausibility_1_to_5": 2, "note": "Film-related candidates are topical, but too generic for a person-answer question."},
    {"sample": 4, "plausibility_1_to_5": 1, "note": "Repeated phrases like 'used' and 'things' are weak distractors."},
    {"sample": 5, "plausibility_1_to_5": 3, "note": "Restaurant phrases match the passage context, but need grammar shaping."},
])
manual_ratings

,sample,plausibility_1_to_5,note
0,1,4,Kodak-related candidates are on-topic and beli...
1,2,3,Kitesurfing/location phrases are relevant but ...
2,3,2,"Film-related candidates are topical, but too g..."
3,4,1,Repeated phrases like 'used' and 'things' are ...
4,5,3,"Restaurant phrases match the passage context, ..."


**Distractor discussion:** the TF-IDF baseline can retrieve on-topic phrases and the cosine-distance score confirms whether the three distractors are diverse. Its main weakness is fluency: extracted phrases can be semantically relevant but not always shaped like polished multiple-choice options. This is why the full project adds stronger Model B filtering and hint/distractor ranking on top of TF-IDF signals.